# 06 — Audit Columns Pattern

Add standard audit columns for traceability.

Process:

DATA
  |
  v
ADD AUDIT COLUMNS
  |
  v
ENRICHED DATA

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("audit-columns-pattern")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions","4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

## Step 1 — Input

In [2]:
schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("amount", DoubleType(), True),
])

rows = [
    ("e1","u1",10.0),
    ("e2","u2",20.0),
]

df = spark.createDataFrame(rows, schema)
df.show()

+--------+-------+------+
|event_id|user_id|amount|
+--------+-------+------+
|      e1|     u1|  10.0|
|      e2|     u2|  20.0|
+--------+-------+------+



## Step 2 — Audit config

In [3]:
pipeline_run_id = "run_2026_01_01_001"
batch_id = "batch_001"

## Step 3 — Add audit columns

In [4]:
with_audit = (
    df
    .withColumn("created_at", F.current_timestamp())
    .withColumn("updated_at", F.current_timestamp())
    .withColumn("pipeline_run_id", F.lit(pipeline_run_id))
    .withColumn("batch_id", F.lit(batch_id))
)

with_audit.show(truncate=False)

+--------+-------+------+--------------------------+--------------------------+------------------+---------+
|event_id|user_id|amount|created_at                |updated_at                |pipeline_run_id   |batch_id |
+--------+-------+------+--------------------------+--------------------------+------------------+---------+
|e1      |u1     |10.0  |2026-05-02 15:08:52.517886|2026-05-02 15:08:52.517886|run_2026_01_01_001|batch_001|
|e2      |u2     |20.0  |2026-05-02 15:08:52.517886|2026-05-02 15:08:52.517886|run_2026_01_01_001|batch_001|
+--------+-------+------+--------------------------+--------------------------+------------------+---------+



## Step 4 — Update scenario

In [5]:
updated = (
    with_audit
    .withColumn("amount", F.col("amount") + 5)
    .withColumn("updated_at", F.current_timestamp())
)

updated.show(truncate=False)

+--------+-------+------+--------------------------+--------------------------+------------------+---------+
|event_id|user_id|amount|created_at                |updated_at                |pipeline_run_id   |batch_id |
+--------+-------+------+--------------------------+--------------------------+------------------+---------+
|e1      |u1     |15.0  |2026-05-02 15:08:52.988553|2026-05-02 15:08:52.988553|run_2026_01_01_001|batch_001|
|e2      |u2     |25.0  |2026-05-02 15:08:52.988553|2026-05-02 15:08:52.988553|run_2026_01_01_001|batch_001|
+--------+-------+------+--------------------------+--------------------------+------------------+---------+



## Step 5 — Control totals

In [6]:
totals = spark.createDataFrame(
    [
        ("input_rows", df.count()),
        ("output_rows", with_audit.count())
    ],
    ["metric","value"]
)

totals.show()

+-----------+-----+
|     metric|value|
+-----------+-----+
| input_rows|    2|
|output_rows|    2|
+-----------+-----+



In [7]:
spark.stop()